This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data. Is there more terms that we are missing? and How to group these terms

# 1. Imports and Data preparation

In [64]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow

# Data processing and analysis
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from typing import Set, List, Union, Dict, Tuple, Optional

In [65]:

sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

Running in local (VSCode) environment


In [66]:
# environment check
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    # For Kaggle # Whole filtered Data
    base_dir =  Path("/kaggle/input/")

    eccontri_path = base_dir / "/eccontri-uniprot-enriched/ECcontri_Uniprot_enriched.parquet"
    pathway_path = base_dir / "gterms/pathways.csv"
    react_path = base_dir / "gterms/reactions.csv"
    # Output for small files
    output_base = Path("/kaggle/working/output_base")
    output_base.mkdir(parents=True, exist_ok=True)
    # Directory to output large files 
    large_dir =  Path("/kaggle/working/")
    # Directory to output large files # eccontris, compilated db

else:  
    # Create output directory if it doesn't exist
    #base dir for small files to git
    base_dir = Path("/home/beatriz/MIC/2_Micro/data_picrust")
    output_base = base_dir / "output_base"
    output_base.mkdir(parents=True, exist_ok=True)            
    # For Vscode # Whole filtered Data large size dir for large files hosted instead in kaggle
    large_dir = Path("/home/beatriz/MIC")
    # Directory to output large files # eccontris, compilated dbs
    output_large = large_dir / "output_large"
    # Whole filtered Data
    eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'
    pathway_path = large_dir / "2_Micro/data_picrust/pathways.csv"
    react_path = large_dir / "2_Micro/data_picrust/reactions.csv"

In [67]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

# 2. Validating terms as real_terms
## 2.1. Checking the terms against the teoretical global_terms

In [68]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [69]:
'''real_terms = validate_terms(ECcontri_Uniprot_enriched,
    [cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])'''

'real_terms = validate_terms(ECcontri_Uniprot_enriched,\n    [cs.metal_terms,\n    cs.corrosion_mechanisms,\n    cs.pathway_categories,\n    cs.organic_categories,\n    cs.corrosion_synergies,\n    cs.functional_categories,\n    cs.corrosion_keyword_groups\n])'

In [70]:
# path to the list of dictionaries
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
    gterms_path = large_dir / 'real_pathways_reactions.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'
    gterms_path = output_large / 'real_pathways_reactions.pkl'

In [71]:
# Saving the new dataframe
#with open(rt_path, 'wb') as f:
#    pickle.dump(real_terms, f)

In [72]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)
# Print in compact format
for category, terms in real_terms.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

'iron': [iron, ferric, heme, iron-sulfur, siderophore, ferritin, ferredoxin, rubredoxin]
'manganese': [mn]
'copper': [copper]
'nickel': [Ni2+]
'cobalt': [cobalt, cobalamin, vitamin B12]
'magnesium': [magnesium]
'calcium': [Ca2+, calcium]
'Mo': [Mo, molybdenum, molybdopterin, molybdenum cofactor]
'V5+': [V5+, vanadium]
'Al3+': [Al3+]
'Cr3+': [Cr3+]
'zinc': [Zn2+, zinc]
'selenium': [selenium, Se, selenocysteine, selenoprotein, selenite, selenate]
'lead': [lead]
'arsenic': [arsenic, arsenate]
'mercury': [mercury, mercuric]
'phosphate': [phosphate, orthophosphate]
'nitrate': [NO3-, nitrate]
'nitrite': [nitrite]
'chloride': [Cl-, chloride]
'sulfate': [sulfate]
'sulfide': [sulfide, h2s]
'thiosulfate': [thiosulfate]
'oxygen': [O2, oxygen, oxidase, superoxide, peroxide]
'hydrogen': [hydrogenase, h2]
'organics': [methane, methane, methanogenesis, formate, formate, formic acid, acetate, acetate, acetic acid, propionate, propionate, propionic acid, butyrate, butyrate, butyric acid, lactate, lacti

## 2.2 Critical review of real terms agains global terms

The process undergone for the scoring system has been iterative and during the first iteration it was noticed that pathways and mechanisms are highly interconnected due to the fact that one bacterium expresses multiple proteins across many pathways and mechanisms. Pathway and mechanism categories exhibit substantial overlap due to multi-protein expression patterns within individual bacterial strains. As a response to this iteration, functional_categories were designed to reconcile this complexity by grouping related processes and make it more about functional metabolism. On a following iteration more modern terms were introduced and diverse terms were tried. Ultimately, it was evident that it was necesary a reality check to validate the terms with the bioinformatics annotations. 
An script was done to critically evaluate the real terms possible to be mined from the compiled database after the enrichment of the data (ECcontri_Uniprot_enriched) with ec_records. The script compared the enriched data with the global terms which teoretically proposed the dictionaries grouped by categories namely: metal_terms, mechanisms, pathways, functional_categories, organic_processes, synergies and keywords.
Analysis of the enriched data's real terms, prompt to redefine the corrosion scoring system by eliminating non-existent terms and consolidating overlapping categorical structures. Yet some theoretical terms are left for teoretical completness. A manual curation was done to reasign categories for efficient computational resource allocation.
The categories to consolidate are: corrosion_synergies,metal_terms, functional_categories, mechanisms and pathways. The categories to remove are: corrosion_keyword_groups and organic_processes. A hierarchy of the categories is stablished, this allows a first term algorithm to prioritize the terms allocated, in order to prevent duplicates.
The scoring takes into account only corrosion_synergies,metal_terms and functional_categories.


# 2.3 Extracting terms from Pathway, ipath and reactions  Finding new real_terms : class BiologicalTermDiscovery
Rather that giving teoretical terms of search, we ectract high frequency terms from the data columns to contrast with the databases and find more information. The df in which those columns were stored is now read and querryied.

Before the terms are compile and arrange on different dictionaries, it was created a script to assest the possible new terms that could be in the enriched df. By using a comprehensive toolkit for discovering and analyzing biological terms from datasets. This class provides methods to extract terms from text data, identify patterns,and discover novel biological terms based on existing validated terms (real_terms).

In [93]:
from collections import Counter, defaultdict
import re
import pandas as pd
from typing import Dict, List, Set, Optional, Tuple
import numpy as np

class EnhancedBiologicalTermDiscovery:
    """
    Enhanced toolkit for discovering corrosion-relevant biological terms from datasets.
    Focuses on mechanistically relevant terms for HVAC/closed water systems.
    """

    def __init__(self, min_term_length: int = 3, stop_words: Optional[Set[str]] = None):
        """
        Initialize with corrosion-focused filtering and scoring.
        """
        self.min_term_length = min_term_length
        self.stop_words = stop_words or {
            'and', 'or', 'the', 'of', 'in', 'to', 'for', 'with', 'by',
            'from', 'at', 'on', 'high', 'general', 'families', 'viral',
            'rna', 'gene', 'direct', 'organics', 'groups', 'ambiguous', 'messenger',
            'iii', 'ii', 'i', 'related', 'associated', 'specific', 'dependent',
            'independent', 'mediated', 'coupled'
        }
        
        # Corrosion-relevant indicators for term scoring
        self.corrosion_indicators = {
            'high_relevance': {
                'iron', 'sulfur', 'sulfate', 'sulfide', 'redox', 'electron', 'oxide',
                'reduction', 'oxidation', 'corrosion', 'biofilm', 'adhesion',
                'chelation', 'binding', 'metal', 'mineral', 'precipitation',
                'dissolution', 'acid', 'organic', 'acetate', 'lactate', 'formate',
                'hydrogenase', 'cytochrome', 'reductase', 'oxidase', 'pili',
                'nanowire', 'extracellular', 'transport', 'respiration'
            },
            'medium_relevance': {
                'carbon', 'nitrogen', 'phosphate', 'metabolism', 'biosynthesis',
                'degradation', 'pathway', 'cycle', 'enzyme', 'protein',
                'membrane', 'cell', 'surface', 'attachment', 'formation'
            },
            'low_relevance': {
                'transcription', 'translation', 'ribosome', 'dna', 'rna',
                'repair', 'stress', 'response', 'regulation', 'signal'
            }
        }
        
        # Mechanistic patterns specific to corrosion processes
        self.mechanistic_patterns = {
            'electron_transfer': ['electron', 'redox', 'cytochrome', 'quinone', 'nadh', 'fadh'],
            'metal_interaction': ['iron', 'metal', 'mineral', 'oxide', 'sulfide', 'chelat'],
            'biofilm_related': ['biofilm', 'adhesion', 'attachment', 'surface', 'matrix', 'eps'],
            'metabolic_products': ['acid', 'acetate', 'lactate', 'sulfide', 'hydrogen', 'organic'],
            'enzymatic_processes': ['ase', 'reductase', 'oxidase', 'dehydrogenase', 'transferase']
        }

    def calculate_corrosion_relevance_score(self, term: str) -> float:
        """
        Calculate relevance score for corrosion research based on term content.
        
        Args:
            term: The term to score
            
        Returns:
            Float score (higher = more relevant to corrosion)
        """
        term_lower = term.lower()
        words = term_lower.split()
        score = 0.0
        
        # Base scoring by relevance categories
        for word in words:
            if any(indicator in word for indicator in self.corrosion_indicators['high_relevance']):
                score += 3.0
            elif any(indicator in word for indicator in self.corrosion_indicators['medium_relevance']):
                score += 1.5
            elif any(indicator in word for indicator in self.corrosion_indicators['low_relevance']):
                score += 0.5
        
        # Bonus for mechanistic patterns
        for pattern_type, indicators in self.mechanistic_patterns.items():
            if any(indicator in term_lower for indicator in indicators):
                score += 2.0
                break  # Only count once per term
        
        # Bonus for specific chemical/biological nomenclature
        if re.search(r'pwy-\d+|rxn-\d+', term_lower):  # Pathway/reaction IDs
            score += 1.5
        
        if re.search(r'\d+-\w+|\w+-\d+', term_lower):  # Chemical nomenclature patterns
            score += 1.0
            
        # Penalty for overly generic terms
        generic_penalties = ['general', 'various', 'other', 'miscellaneous', 'unspecified']
        if any(penalty in term_lower for penalty in generic_penalties):
            score -= 2.0
            
        return score

    def extract_all_terms(self, series: pd.Series, max_ngram_length: int = 6) -> Counter:
        """
        Extract all terms (including multi-word phrases/n-grams) from a pandas Series.
        Preserves complex biological pathway names.

        Args:
            series: Pandas Series containing text data.
            max_ngram_length: Maximum length of n-grams (phrases) to extract.

        Returns:
            Counter object with term frequencies.
        """
        terms = Counter()

        for text in series.dropna():
            if isinstance(text, str):
                # Normalize spaces and convert to lowercase
                normalized_text = re.sub(r'\s+', ' ', text).lower()

                # Split text into potential segments based on strong punctuation (excluding hyphens/underscores)
                # This allows terms like "N10-formyl-tetrahydrofolate biosynthesis" to be processed as one chunk
                segments = re.split(r'[.,;|\n\t\(\)\[\]]+', normalized_text)

                for segment in segments:
                    segment = segment.strip()
                    if len(segment) < 3:
                        continue
                        
                    # First, try to capture the whole segment if it's a valid biological term
                    cleaned_segment = re.sub(r'^[^\w\d\-_]+|[^\w\d\-_]+$', '', segment)

    def identify_functional_clusters(self, terms: Counter, existing_categories: Dict) -> Dict[str, List[str]]:
        """
        Cluster new terms into potential functional categories based on similarity to existing ones.
        
        Args:
            terms: Counter of discovered terms
            existing_categories: Current functional categories dictionary
            
        Returns:
            Dictionary mapping potential categories to new terms
        """
        clusters = defaultdict(list)
        
        # Create keyword profiles for existing categories
        category_profiles = {}
        for category, data in existing_categories.items():
            if isinstance(data, dict) and 'terms' in data:
                # Extract key indicators from existing terms
                indicators = set()
                for term in data['terms']:
                    term_words = term.lower().split()
                    for word in term_words:
                        if len(word) > 3:  # Focus on meaningful words
                            indicators.add(word)
                category_profiles[category] = indicators
        
        # Classify new terms
        for term, freq in terms.items():
            if freq >= 5:  # Minimum frequency threshold
                term_words = set(term.lower().split())
                
                best_category = None
                best_overlap = 0
                
                for category, indicators in category_profiles.items():
                    overlap = len(term_words.intersection(indicators))
                    if overlap > best_overlap:
                        best_overlap = overlap
                        best_category = category
                
                if best_overlap >= 1:  # At least one word overlap
                    clusters[best_category].append(term)
                else:
                    # Create new potential categories for unmatched terms
                    corr_score = self.calculate_corrosion_relevance_score(term)
                    if corr_score >= 3.0:
                        clusters['high_corrosion_potential'].append(term)
                    elif corr_score >= 1.5:
                        clusters['medium_corrosion_potential'].append(term)
        
        return dict(clusters)

    def discover_novel_terms(self, df: pd.DataFrame,
                           metal_terms: Dict[str, List[str]],
                           corrosion_synergies: Dict[str, List[str]],
                           functional_categories: Dict[str, Dict],
                           text_columns: List[str],
                           min_frequency: int = 10) -> List[str]:
        """
        Main method to discover novel biological terms using enhanced corrosion-relevance filtering.
        
        Args:
            df: DataFrame containing the biological data
            metal_terms: Dictionary of metal-related terms
            corrosion_synergies: Dictionary of corrosion synergy terms
            functional_categories: Dictionary of functional category terms
            text_columns: List of column names containing text data
            min_frequency: Minimum frequency threshold for terms

        Returns:
            List of new terms discovered
        """
        # Combine all real terms dictionaries
        all_real_dicts = {
            **metal_terms,
            **corrosion_synergies,
            **functional_categories
        }

        # Extract terms using enhanced method that preserves long pathway names
        all_discovered_terms = Counter()
        
        for col in text_columns:
            if col in df.columns:
                column_terms = self.extract_all_terms(df[col], max_ngram_length=6)
                all_discovered_terms.update(column_terms)

        # Create flat set of all real terms (lower-cased)
        all_real_terms = set()
        for terms_data in all_real_dicts.values():
            if isinstance(terms_data, list):
                all_real_terms.update(term.lower() for term in terms_data)
            elif isinstance(terms_data, dict):
                if 'terms' in terms_data:
                    all_real_terms.update(term.lower() for term in terms_data['terms'])
                else:
                    for subterms_list in terms_data.values():
                        if isinstance(subterms_list, list):
                            all_real_terms.update(term.lower() for term in subterms_list)

        new_terms_set = set()

        # Sort terms by length first (prioritize longer, more specific terms), then corrosion relevance, then frequency
        sorted_discovered_terms = sorted(
            all_discovered_terms.items(),
            key=lambda item: (-len(item[0].split()), -len(item[0]), self.calculate_corrosion_relevance_score(item[0]), item[1]),
            reverse=True
        )

        for term, freq in sorted_discovered_terms:
            if (term not in all_real_terms and
                freq >= min_frequency and
                len(term) >= self.min_term_length and
                not term.isdigit() and
                not term.startswith('br:ko') and
                not re.match(r'^[a-z]{2}:\w+\d+', term)):

                # Apply corrosion relevance filtering - but be more lenient for long pathway names
                relevance_score = self.calculate_corrosion_relevance_score(term)
                term_words = term.split()
                
                # For long pathway names, be more lenient with scoring
                is_pathway_name = len(term_words) >= 3 and any(keyword in term for keyword in ['biosynthesis', 'degradation', 'metabolism', 'fermentation', 'pathway', 'superpathway'])
                
                min_score = 1.0 if is_pathway_name else 2.0
                
                if relevance_score >= min_score:
                    
                    # Skip single generic words unless they're specific identifiers
                    if len(term_words) == 1 and term in self.stop_words and \
                       not (term.startswith('pwy-') or term.startswith('rxn-')):
                        continue

                    # Skip if all words are generic - but be lenient for pathway names
                    if len(term_words) > 1 and all(word in self.stop_words for word in term_words) and not is_pathway_name:
                        continue

                    # For very long terms (like pathway names), be less strict about substring filtering
                    if len(term_words) >= 4:
                        new_terms_set.add(term)
                        continue

                    # Check for redundant substrings
                    is_redundant_substring = False
                    for existing_added_term in list(new_terms_set):
                        if term != existing_added_term and term in existing_added_term and \
                           (freq <= all_discovered_terms[existing_added_term] or len(term_words) < len(existing_added_term.split())):
                            is_redundant_substring = True
                            break
                    if is_redundant_substring:
                        continue

                    new_terms_set.add(term)

        return sorted(list(new_terms_set))

    def suggest_category_assignments(self, discovered_terms: List[str], 
                                   existing_categories: Dict[str, Dict]) -> Dict[str, List[str]]:
        """
        Suggest which existing categories new terms should be added to.
        
        Args:
            discovered_terms: List of new terms to categorize
            existing_categories: Current functional categories
            
        Returns:
            Dictionary mapping category names to suggested new terms
        """
        suggestions = defaultdict(list)
        
        for term in discovered_terms:
            term_lower = term.lower()
            best_matches = []
            
            for category, data in existing_categories.items():
                if isinstance(data, dict) and 'terms' in data:
                    match_score = 0
                    
                    # Check for direct word matches
                    term_words = set(term_lower.split())
                    category_words = set()
                    for existing_term in data['terms']:
                        category_words.update(existing_term.lower().split())
                    
                    word_overlap = len(term_words.intersection(category_words))
                    match_score += word_overlap * 2
                    
                    # Check for pattern matches
                    for pattern_type, indicators in self.mechanistic_patterns.items():
                        if any(indicator in term_lower for indicator in indicators):
                            if any(indicator in ' '.join(data['terms']).lower() for indicator in indicators):
                                match_score += 1
                    
                    if match_score > 0:
                        best_matches.append((category, match_score))
            
            # Sort by match score and take top matches
            best_matches.sort(key=lambda x: x[1], reverse=True)
            
            if best_matches and best_matches[0][1] >= 2:  # Minimum threshold
                suggestions[best_matches[0][0]].append(term)
            else:
                suggestions['needs_new_category'].append(term)
        
        return dict(suggestions)

In [ ]:
metal_terms = {	
'iron': ['Fe2+', 'Fe3+', 'iron', 'ferrous', 'ferric', 'heme', 'iron-sulfur', 'rust', 'ochre', 'iron oxide', 'siderophore', 'ferritin'],	
'manganese': ['Mn2+',  'manganese', 'mn', 'manganous', 'manganic', 'manganese oxidation', 'manganese oxide', 'MnO2'],	
'copper': ['Cu+', 'Cu2+', 'copper', 'cupric', 'cuprous', 'copper oxide', 'copper corrosion'],	
'nickel': ['Ni2+', 'nickel', 'nickelous', 'nickel oxidation', 'nickel reduction'],	
'cobalt': ['Co2+',  'cobalt', 'cobaltous', 'cobalamin', 'vitamin B12'],	
'magnesium': ['Mg2+', 'magnesium', 'magnesium oxide'],	
'calcium': ['Ca2+', 'calcium', 'calcium carbonate', 'calcite', 'calcium precipitation'],	
'Mo': ['Mo',  'molybdenum', 'molybdopterin', 'molybdenum cofactor'],	
'V5+': ['V5+', 'vanadium', 'vanadate', 'vanadyl'],	
'Al3+': ['Al3+', 'aluminum', 'aluminate', 'aluminum oxide'],	
'Cr3+': ['Cr3+', 'Cr6+', 'chromium', 'chromate', 'dichromate', 'chromium oxide'],	
'zinc': ['Zn2+', 'zinc', 'zinc finger', 'zinc oxide'],	
'sodium': ['Na+', 'sodium', 'NaCl', 'sodium transport', 'sodium gradient'],	
'potassium': ['K+', 'potassium', 'KCl', 'potassium transport', 'potassium channel'],	
'selenium': ['selenium', 'Se', 'selenocysteine', 'selenoprotein', 'selenite'],	
'barium': ['Ba2+', 'barium', 'barium sulfate', 'barite'],	
'strontium': ['Sr2+', 'strontium', 'strontium carbonate', 'strontium sulfate'], 	
'lead': ['Pb2+', 'Pb4+', 'lead', 'plumbous', 'plumbic', 'lead oxide'], 	
'arsenic': ['As3+', 'As5+', 'arsenic', 'arsenite', 'arsenate', 'arsenic oxidation'], 	
'mercury': ['Hg2+', 'Hg+', 'mercury', 'mercuric', 'mercurous', 'mercury methylation'], 	
'phosphate': ['HPO4-2', 'PO4-3', 'phosphate', 'phosphates'],	
'nitrogen':['NO3-', 'nitrate', 'nitrates','NO2-', 'nitrite', 'nitrites'],	
'chloride': ['Cl-', 'chloride', 'chlorine'],	
'sulphate':['SO4-2', 'sulfate', 'sulfates', 'S', 'sulfide', 'sulfides', 'H2S', 'hydrogen sulfide', 'S2O3-2', 'thiosulfate'],		
'oxygen': ['O2', 'oxygen', 'oxidase'],	
'hydrogen': ['H2', 'hydrogen', 'hydrogenase', 'hydrogen uptake', 'hydrogen evolution'],	
'organics': ['methane', 'CH4', 'methanogenic', 'methanogenesis', 'formate','formic acid', 'HCOO-', 'acetate', 'acetic acid', 'CH3COO-', 'propionate','propionate', 'propionic acid', 'butyrate', 'butyric acid','lactate', 'lactic acid', 'mercaptans', 'mercaptan', 'thiol', 'methanethiol', 'ethanethiol', 'H2S', 'h2s', 'alcohol', 'ethanol', 'methanol', 'propanol', 'alcohol']
}	

## Synergies
The dictionary allows to retrieve the explicit menton of common synergistic compunds  (e.g., "iron sulfide" implies Fe-S interaction, "stainless steel" implies Cr-Fe). This approach struggles to infer synergy from disparate but co-occurring terms across different fields. For example, if 'Iron' is listed in clean_metals and 'acid attack' is a corrosion_mechanism or 'acidic' is in biological_function, the current system won't automatically flag an "Iron-Acid Synergy" unless the exact combined phrase "iron acid" is present in all_text. Additional nuance must be taken into account such as "Fe-S cluster" is often a structural component of an enzyme (a cofactor). While relevant to iron and sulfur, its presence doesn't always mean the enzyme is directly involved in a corrosive Fe-S synergy in the external environment. It means the enzyme uses Fe-S within its own structure. This highlights the need for careful semantic interpretation. At this point a context relevant rule was stablished, so that several columns would ultimately be look up in order to check for synergies. For instance if an enzyme interacts with a metal, has a specific functional category (e.g., a reductase for sulfur compounds), and is found in an environment susceptible to corrosion, that's a strong synergistic indicator. 

In [ ]:
corrosion_synergies= {
'Fe-S': ['iron_sulfur', 'Fe-S','iron sulfide','FeS', 'Fe-S cluster'],	
'Fe-Cl': ['iron chloride', 'FeCl', 'iron halide', 'ferric chloride'],	
'Fe-C': ['iron carbon', 'FeC', 'iron carbonate', 'siderite'],	
'Cu-Fe': ['copper iron', 'Cu-Fe', 'bimetallic', 'galvanic couple'],	
'Mn-Fe': ['manganese iron', 'Mn-Fe', 'iron manganese oxide'],
'Ni-Fe': ['Ni-Fe'],  	
'Cr-Fe': ['chromium iron', 'Cr-Fe', 'stainless steel', 'chromium passivation'], 	
'Al-Cu': ['aluminum copper', 'Al-Cu', 'aluminum brass', 'galvanic corrosion'], 	
'Zn-Fe': ['zinc iron', 'Zn-Fe', 'galvanized steel', 'sacrificial anode'],	
'Fe-CO3': ['iron carbonate', 'siderite', 'bicarbonate corrosion', 'carbonate scaling'],	
'Fe-SO4': ['iron sulfate', 'sulfate corrosion', 'gypsum formation'],	
'Fe-Ox': ['iron oxalate', 'oxalate corrosion', 'organic acid attack', 'oxidation corrosion'],	
'Fe-Ac': ['iron acetate', 'acetate corrosion', 'organic acid attack', 'oxidation corrosion']}	
	
functional_categories =	{
'o2_consumption': {'terms': ['o2_consumption', 'aerobic_respiration', 'oxygen reduction', 'oxygen consumption', 'cytochrome oxidase', 'oxidase', 'terminal oxidase',  'oxygen reductase',  'superoxide dismutase',  'catalase',  'oxidative stress', 'oxygen sensor',  'oxygen tolerance', 'oxygen consum', 'oxygen scavenging', 'oxygen stress', 'oxidative phosphorylation', 'NADH dehydrogenase', 'CYTOCHROME-C-OXIDASE','ATPSYN-RXN', "TCA cycle IV (2-oxoglutarate decarboxylase)","TCA cycle V (2-oxoglutarate:ferredoxin oxidoreductase)",  "TCA cycle VI (obligate autotrophs)", "TCA cycle VIII (helicobacter)", "superpathway of glyoxylate bypass and TCA"], 'score': 1.2, 'justification': 'Geesey, G.G., Bremer, P.J. (1990). Biofouling of engineered water systems. Biotechnol Bioeng, 36(10):1039-1046'},# VERY HIGH for HVAC - oxygen depletion creates aggressive conditions
'nitrogen_metabolism':  {'terms': ['nitrate_reduction', 'nitrite_reduction', 'denitrification', 'nitrification', 'nitrate respiration', 'nitrite respiration', 'nitrous oxide reduction', 'ammonia oxidation', 'anammox', 'nitrogen fixation', 'ammonification', 'nitrogen metabolism',  'nitrate',  'nitrite','dissimilatory nitrate reduction',  'nitrite reductase', 'nitrate reductase', "L-arginine biosynthesis II (acetyl cycle)", "L-ornithine biosynthesis", "L-histidine biosynthesis", "L-methionine biosynthesis III", "L-isoleucine biosynthesis I (from threonine)", "L-isoleucine biosynthesis II", "L-isoleucine biosynthesis III", "L-isoleucine biosynthesis IV", "L-lysine biosynthesis I", "L-lysine biosynthesis III", "L-lysine biosynthesis VI", "L-valine biosynthesis", "L-tryptophan biosynthesis", "superpathway of L-isoleucine biosynthesis I", "superpathway of L-phenylalanine biosynthesis", "superpathway of L-tyrosine biosynthesis", "superpathway of L-threonine biosynthesis" ], 'score': 1.0,'justification': 'Flemming, H.C. (1996). Economically relevant microorganisms in technical systems. Materials and Corrosion, 47(7):391-398 and  Washizu, N., et al. (2004). MIC by nitrate-reducing bacteria in water injection systems. Corrosion, 60(4):336-342'}, # Moderate for HVAC - affects redox conditions	
'iron_metabolism': {'terms': ['corrosion', 'MIC', 'ferric reduc', 'SRB', 'ocre', 'iron_oxide', 'iron_deposit', 'metal oxide', 'ochre formation', 'iron oxide deposits', 'iron precipitation', 'rust formation', 'iron oxid', 'ferrous oxid', 'ferric', 'iron uptake', 'iron transport', 'iron storage',  'iron homeostasis', 'siderophore production', 'iron_sulfur_redox', 'ferredoxin',  'rubredoxin', 'ferritin', 'bacterioferritin',  'PWY-7221',  'PWY-7219', 'HEME-BIOSYNTHESIS-II',  'P125-PWY', 'iron mobilization', 'iron immobilization', 'ferrihydrite', 'goethite', 'magnetite', 'hematite', 'iron mineral', 'biogenic iron oxides', 'stalactite formation', 'ochre mats', 'superpathway of tetrahydrofolate biosynthesis and salvage',  "heme biosynthesis II (anaerobic)", "tetrapyrrole biosynthesis I (from glutamate)", "tetrapyrrole biosynthesis II (from glycine)","flavin biosynthesis I (bacteria and plants)","chorismate biosynthesis I", "chorismate biosynthesis from 3-dehydroquinate"], 'score': 1.5, 'justification': 'Beech, I.B., Gaylarde, C.C. (1999). Recent advances in the study of biocorrosion. Rev Microbiol, 30(3):177-190. SRB critical in closed water systems AND Cornell, R.M., Schwertmann, U. (2003). The Iron Oxides. Green rust and iron-organic complexes indicate active corrosion processes'},# represents active Fe-organic complexation '},	# HIGHEST for HVAC - SRB major problem in closed loops
'sulfur_metabolism': {'terms': ['sulphur_metabolism', 'sulfur_metabolism', 'sulfate reduc', 'sulfite', 'thiosulfate', 'sulfur oxidation', 'SRB', 'dsrAB', 'APS reductase', 'sulfide','quinone oxidoreductase', 'dissimilatory sulfate reduction', 'sulfur globules', 'elemental sulfur', 'polysulfide metabolism', 'sulfur granules', 'PWY-6932', 'SO4ASSIM-PWY', 'SULFATE-CYS-PWY', 'sulfide_production', 'sulfonate',  'sulfur_reduction', 'desulfovibrio', 'sulfur disproportionation', 'sulfate-reducing bacteria', 'sulfur respiration', "sulfate reduction I (assimilatory)", "superpathway of sulfate assimilation and cysteine biosynthesis"], 'score': 1.5, 'justification': 'Sulfide causes direct corrosive attack on iron and steel surfaces'},# Direct corrosive attack but secondary to SRB iron-sulfur redox
'h2_consumption': {'terms': ['h2_consumption', 'hydrogenase', 'hydrogen uptake', 'hydrogen consumption', 'h2', 'H2 oxidation', 'H2ase', 'hydrogen metabolism', 'hydrogen production',  'FeFe-hydrogenase',  'NiFe-hydrogenase',  'hydrogen evolution',  'hydrogen cycling',  'H2 sensing',  'proton reduction', "methylerythritol phosphate pathway I", "methylerythritol phosphate pathway II"], 'score': 0.7, 'justification': 'Javaherdashti, R. (2008). Microbiologically Influenced Corrosion: An Engineering Insight. Springer and Enning, D., Garrelfs, J. (2014). Corrosion of iron by sulfate-reducing bacteria: new views of an old problem. Appl Environ Microbiol, 80(4):1226-1236'},# H2 consumption by SRB accelerates corrosion through cathodic depolarization	
'direct_eet':  {'terms': ['cytochrome c oxidase', 'quinol oxidase', 'NADH:quinone oxidoreductase', 'succinate dehydrogenase', 'fumarate reductase', 'cytochrome', 'electron transfer', 'electron transport', 'conductive pili', 'nanowire', 'mtrABC', 'omc', 'omcS', 'oxidoreductase', 'redox', 'reductase', 'oxidase', 'electron conduit', 'direct electron transfer', 'deet', 'c-type cytochrome', 'multi-heme cytochrome', 'flavin', 'electron shuttle','histidine kinase','PROTEIN-KINASE-RXN', 'L-arginine biosynthesis II', 'chorismate biosynthesis I', 'superpathway of branched amino acid biosynthesis', 'superpathway of aromatic amino acid biosynthesis', 'L-lysine biosynthesis I', 'L-ornithine biosynthesis'], 'score': 1.3, 'justification':  'Jones, D.A. (1996). Principles and Prevention of Corrosion, 2nd ed. All corrosion fundamentally involves electron transfer'},# Important but secondary to chemical mechanisms in HVAC	
'carbon_metabolism': {'terms': ['carbon_metabolism', 'carbon fixation', 'carbon utilization', 'carbohydrate metabolism', 'glycolysis', 'TCA cycle', 'carbon flux', 'carbon assimilation', 'pentose phosphate pathway', 'gluconeogenesis', 'Calvin cycle', 'reductive acetyl-CoA pathway', 'carbon monoxide dehydrogenase', 'hydrocarbon degradation', 'aromatic degradation', 'alcohol metabolism', 'organic matter degradation', 'VFA production', 'propionate', 'butyrate', 'valerate', 'caproate', 'ACETYL-COA-ACETYLTRANSFER-RXN','METHYLACETOACETYLCOYTHIOL-RXN','ACETOLACTSYN-RXN','ACETOOHBUTSYN-RXN','ACETYL-COA-CARBOXYLTRANSFER-RXN','ACYLCOASYN-RXN','N10-formyl-tetrahydrofolate biosynthesis', 'glycolysis III','Calvin-Benson-Bassham cycle', 'gluconeogenesis', "glycolysis I (from glucose 6-phosphate)", "glycolysis II (from fructose 6-phosphate)","glycolysis III (from glucose)","gluconeogenesis I","pentose phosphate pathway (non-oxidative branch)","Calvin-Benson-Bassham cycle","pyruvate fermentation to isobutanol (engineered)","superpathway of branched amino acid biosynthesis","superpathway of aromatic amino acid biosynthesis","superpathway of adenosine nucleotides de novo biosynthesis I","superpathway of adenosine nucleotides de novo biosynthesis II","superpathway of guanosine nucleotides de novo biosynthesis I","superpathway of guanosine nucleotides de novo biosynthesis II"  ], 'score': 0.5,'justification': 'Pope, D.H. (1986). A study of microbiologically influenced corrosion in nuclear power plants. Electric Power Research Institute'},
'indirect_eet': {'terms': ['shuttle', 'mediator', 'redox mediator', 'electron shuttle', 'flavin', 'quinone', 'humic substance'], 'score': 0.5,'justification': 'Enning, D., Garrelfs, J. (2014). Corrosion of iron by sulfate-reducing bacteria: new views of an old problem. Appl Environ Microbiol, 80(4):1226-1236'},
'organic_acid_metabolism': {'terms':  ['acetate', 'acetic acid', 'acetyl', 'acetate metabolism', 'acetate production', 'oxalate', 'oxalic acid', 'oxalate metabolism', 'oxalate production', 'organic acid', 'fatty acid', 'butyric acid', 'butyrate', 'propionate', 'propionic acid', 'carboxylic acid', 'lactate', 'lactic acid', 'formate', 'formic acid', 'citrate', 'citric acid', 'succinate', 'succinic acid', 'fumarate', 'fumaric acid', 'malate', 'malic acid', 'pyruvate', 'pyruvic acid', 'acidification', 'fermentation', 'CENTFERM-PWY', 'FERMENTATION-PWY', 'GLYCOLYSIS', 'PWY-5100', 'GALACTUROCAT-PWY', 'fatty acid β-oxidation I', 'fatty acid elongation',"fatty acid salvage","stearate biosynthesis II (bacteria and plants)","palmitoleate biosynthesis I (from (5Z)-dodec-5-enoate)","cis-vaccenate biosynthesis","oleate biosynthesis IV (anaerobic)","gondoate biosynthesis (anaerobic)","mycolate biosynthesis"], 'score': 1.4, 'justification': 'Videla, H.A., Herrera, L.K. (2005). Microbiologically influenced corrosion: looking to the future. Int Microbiol, 8(3):169-180'}, # VERY HIGH for HVAC - organic acids major issue	
'metal binding / chelation': {'terms': ['metal_chelation', 'metal_binding', 'siderophore', 'complexation', 'iron chelation', 'enzymatic_metal_oxid', 'peroxidase',  'chelator', 'metallophore', 'iron complex', 'metal transport', 'metal oxide', 'iron oxide deposits', 'metal deposition', 'metal solubilization', 'mineral dissolution', 'mineral precipitation', 'chelation', 'metal complexation', 'metal sequestration' , 'metal_organic_interaction', 'metal organic', 'metal homeostasis', 'organometallic',  'iron uptake', 'metal uptake', 'metalloprotein',  'iron-sulfur cluster', 'metal coordination', 'ferric reductase', 'ferrous oxidase', 'metal homeostasis', 'mineral dissolution', 'mineral precipitation', 'copper reduction', 'nickel oxidation', 'chromium reduction', 'crystal nucleation', 'metal immobilization', "coenzyme A biosynthesis I","pantothenate and coenzyme A biosynthesis I","phosphopantothenate biosynthesis I","NAD biosynthesis I (from aspartate)","thiamin salvage II"], 'score': 1.2, 'justification': 'Herrera, L.K., Videla, H.A. (2009). Role of iron-reducing bacteria in corrosion and protection of carbon steel. Int Biodeterior Biodegradation, 63(7):891-895'},# Important in closed loops
'biofilm_formation': {'terms': ['biofilm_formation', 'metal_chelation', 'quorum_sensing', 'extracellular_matrix', 'exopolysaccharide', 'EPS production', 'EPS', 'surface_disruption', 'polysaccharide', 'adhesin', 'biofilm', 'EPS', 'extracellular polymeric substance', 'curli', 'exopolymer','extracellular matrix', 'adhesion', 'colonization', 'attachment', 'surface', 'adherence', 'biofilm maturation', 'biofilm regulation', 'biofilm dispersion', 'cell-cell adhesion', 'surface attachment', 'polysaccharide biosynthesis', 'cell aggregation', 'matrix production', 'pellicle', 'floc formation', 'COLANSYN-PWY', 'EXOPOLYSACC-PWY', 'GLUCOSE1PMETAB-PWY', 'alginate', 'cellulose', 'lipid metabolism', 'fatty acid synthesis', 'fatty acid degradation', 'biosurfactant', 'VFA', 'volatile fatty acid', 'propionate', 'butyrate', 'oleaginous', 'lipid accumulation', '3-oxoacyl', '3-oxoacyl-(acyl-carrier-protein)','3-oxocerotoyl-[acp] reductase','3-oxo-cis-Δ7-tetradecenoyl-[acp] reductase','3-oxo-cis-Δ9-hexadecenoyl-[acp] reductase','3-oxo-glutaryl-[acp] methyl ester reductase','3-oxo-pimeloyl-[acp] methyl ester reductase','3-oxo-docosapentaenoyl [acp][c]','(5Z)-3-oxo-tetradec-5-enoyl-[acyl-carrier-protein] reductase','(7Z)-3-oxo-hexadec-7-enoyl-[acp] reductase','(9Z)-3-oxo-octadec-9-enoyl-[acp] reductase','(11Z)-3-oxo-icos-11-enoyl-[acp] reductase','acetoacetyl-[acyl-carrier protein] reductase','3-hydroxyhexanoyl-[acyl-carrier protein] reductase','3-oxo-octanoyl-[acyl-carrier protein] reductase','3-oxo-decanoyl-[acyl-carrier protein] reductase','LINOLENOYL-RXN', 'coenzyme A biosynthesis I', "peptidoglycan biosynthesis I (meso-diaminopimelate containing)","peptidoglycan biosynthesis III (mycobacteria)","UDP-N-acetylmuramoyl-pentapeptide biosynthesis I (meso-diaminopimelate containing)","UDP-N-acetylmuramoyl-pentapeptide biosynthesis II (lysine-containing)","dTDP-L-rhamnose biosynthesis I","O-antigen building blocks biosynthesis (E. coli)","phosphatidylglycerol biosynthesis I (plastidic)","phosphatidylglycerol biosynthesis II (non-plastidic)","CDP-diacylglycerol biosynthesis I","CDP-diacylglycerol biosynthesis II"], 'score': 1.3, 'justification': 'Borenstein, S.W. (1994). Microbiologically Influenced Corrosion Handbook. Industrial Press. Biofilms critical in closed water systems'},#  biofilms major problem in closed systems	
'manganese_processes': {'terms': ['manganese_reduction', 'mn_redox', 'manganese oxidation', 'manganese oxide',  'pyrolusite',  'birnessite',  'manganese cycling',  'manganese mineral',  'manganese transport', 'Mn-oxide formation', 'Mn-oxide reduction', 'Mn precipitation', 'Mn dissolution'],'score': 1.0,'justification': 'Tebo, B.M., et al. (2004). Biogenic manganese oxides: properties and mechanisms of formation. Annu Rev Earth Planet Sci, 32:287-328'}, 
'methanogenesis': {'terms': ['methanogenesis', 'methanobacterium', 'archaea', 'methane production', 'methyl-coenzyme M reductase', 'methanogenic', 'coenzyme F420',  'methyl-H4MPT', 'CO2 reduction', 'acetoclastic methanogenesis'],'score': 0.6,'justification': 'Mori, K., et al. (2010). Methanogens in microbiologically influenced corrosion: a review. Microorganisms, 8(7):995'},# REDUCED for HVAC - uncommon in aerobic systems	
'fumarate_formation': {'terms': ['fumarate', 'propionibacterium'],  'score': 0.5},# Lower priority in HVAC. 
'halogen_related': {'terms': ['halogen', 'chloride', 'bromide', 'iodide', 'fluoride', 'halide', 'dehalogenation', 'haloperoxidase', 'haloacid', 'chlorination', 'bromination', 'organohalide', 'halomethane', 'haloalkane', 'organohalide', 'halotolerance', 'salt tolerance', 'halophilic', 'chloride transport', 'halide channel', 'chloride attack', 'chloride-induced corrosion', 'pitting initiation', 'chloride penetration', 'halide corrosion', 'perchlorate reduction', 'halorespiration', 'organohalide'],'score': 0.7, 'justification': 'Marcus, P., Oudar, J. (1995). Corrosion Mechanisms in Theory and Practice. Marcel Dekker. Chloride ions initiate pitting corrosion'},# REDUCED for HVAC - less critical in closed systems vs marine	
'ph_modulation': {'terms': ['acid', 'alkaline', 'proton pump', 'pH homeostasis', 'pH stress', 'acid tolerance', 'alkaline tolerance', 'proton motive force', 'pH regulation', 'acidic environment', 'alkaline environment', 'acid resistance', 'proton antiporter', 'proton generation', 'low pH', 'pH buffering', 'pH gradient', 'urease', 'ammonification', 'ammonia production', 'alkali production'], 'score': 0.5,'justification': 'Videla, H.A., Herrera, L.K. (2005). Microbiologically influenced corrosion: looking to the future. Int Microbiol, 8(3):169-180'},	
'phosphorus_metabolism': {'terms': ['phosphate transport',  'polyphosphate', 'phosphite oxidation', 'organophosphonate metabolism',  "UMP biosynthesis", "pyrimidine deoxyribonucleotides de novo biosynthesis I","pyrimidine deoxyribonucleotide phosphorylation","superpathway of pyrimidine nucleobases salvage","superpathway of pyrimidine deoxyribonucleotides de novo biosynthesis","adenosine ribonucleotides de novo biosynthesis","adenosine deoxyribonucleotides de novo biosynthesis II","guanosine ribonucleotides de novo biosynthesis","guanosine deoxyribonucleotides de novo biosynthesis II","superpathway of pyrimidine ribonucleotides de novo biosynthesis","superpathway of pyrimidine deoxyribonucleotides de novo biosynthesis (E. coli)"], 'score': 0.5,'justification': 'Beech, I.B., Sunner, J. (2004). Biocorrosion: towards understanding interactions between biofilms and metals. Curr Opin Biotechnol, 15(3):181-186'},	
'mic' : {'terms': ['antimicrobial production', 'competitive exclusion', 'corrosion inhibition', 'fungal metabolism', 'archaeal metabolism', 'extremophile', '1.11.1.15-RXN', 'GSHTRAN-RXN','GST-RXN'], 'score': -1.5,'justification': 'Little, B.J., Lee, J.S. (2007). Microbiologically Influenced Corrosion. Wiley-Interscience. Protective mechanisms against corrosion'},	
'temp_response': {'terms':['heat shock', 'cold shock', 'temperature response', 'thermophilic', 'psychrophilic', 'mesophilic', 'thermal adaptation', 'temperature stress', 'heat stress protein','cold stress protein', 'thermal stability', 'thermotolerance' ,'osmotic stress', 'desiccation tolerance'],'score': 0.2, 'justification': 'Beech, I.B., Sunner, J. (2004). Biocorrosion: towards understanding interactions between biofilms and metals. Curr Opin Biotechnol, 15(3):181-186'},
'enzymatic_metal_oxid': {'terms': ['metalloenzyme', 'enzyme-catalyzed oxidation', 'peroxidase', 'laccase', 'oxidoreductase activity', 'enzyme-mediated corrosion'], 'score': 0.8,'justification': 'Herrera, L.K., Videla, H.A. (2009). Role of iron-reducing bacteria in corrosion and protection of carbon steel. Int Biodeterior Biodegradation, 63(7):891-895'},
'dealloying_mechanisms': {'terms': ['selective corrosion', 'dezincification', 'dealuminification', 'preferential dissolution', 'parting'], 'score': 0.6, 'justification': ''},
'exoelectrogenesis': {'terms': ['exoelectrogen', 'electrochemically active bacteria', 'EAB', 'extracellular respiration', 'electrode respiration'], 'score': 0.9, 'justification': ''}
}

In [ ]:
'o2_consumption','nitrogen_metabolism':  {'terms': ['nitrate_reduction', 'nitrite_reduction', 'denitrification', 'nitrification', 'nitrate respiration', 'nitrite respiration', 'nitrous oxide reduction', 'ammonia oxidation', 'anammox', 'nitrogen fixation', 'ammonification', 'nitrogen metabolism',  'nitrate',  'nitrite','dissimilatory nitrate reduction',  'nitrite reductase', 'nitrate reductase', "L-arginine biosynthesis II (acetyl cycle)", "L-ornithine biosynthesis", "L-histidine biosynthesis", "L-methionine biosynthesis III", "L-isoleucine biosynthesis I (from threonine)", "L-isoleucine biosynthesis II", "L-isoleucine biosynthesis III", "L-isoleucine biosynthesis IV", "L-lysine biosynthesis I", "L-lysine biosynthesis III", "L-lysine biosynthesis VI", "L-valine biosynthesis", "L-tryptophan biosynthesis", "superpathway of L-isoleucine biosynthesis I", "superpathway of L-phenylalanine biosynthesis", "superpathway of L-tyrosine biosynthesis", "superpathway of L-threonine biosynthesis" ], 'score': 1.3,'justification': 'Flemming, H.C. (1996). Economically relevant microorganisms in technical systems. Materials and Corrosion, 47(7):391-398 and  Washizu, N., et al. (2004). MIC by nitrate-reducing bacteria in water injection systems. Corrosion, 60(4):336-342'}, # Moderate for HVAC - affects redox conditions	
'iron_metabolism', 'sulfur_metabolism': {'terms': ['sulphur_metabolism', 'sulfur_metabolism', 'sulfate reduc', 'sulfite', 'thiosulfate', 'sulfur oxidation', 'SRB', 'dsrAB', 'APS reductase', 'sulfide','quinone oxidoreductase', 'dissimilatory sulfate reduction', 'sulfur globules', 'elemental sulfur', 'polysulfide metabolism', 'sulfur granules', 'PWY-6932', 'SO4ASSIM-PWY', 'SULFATE-CYS-PWY', 'sulfide_production', 'sulfonate',  'sulfur_reduction', 'desulfovibrio', 'sulfur disproportionation', 'sulfate-reducing bacteria', 'sulfur respiration', "sulfate reduction I (assimilatory)", "superpathway of sulfate assimilation and cysteine biosynthesis"], 'score': 1.5, 'justification': 'Sulfide causes direct corrosive attack on iron and steel surfaces'},# Direct corrosive attack but secondary to SRB iron-sulfur redox
'h2_consumption','direct_eet':  {'terms': ['cytochrome c oxidase', 'quinol oxidase', 'NADH:quinone oxidoreductase', 'succinate dehydrogenase', 'fumarate reductase', 'cytochrome', 'electron transfer', 'electron transport', 'conductive pili', 'nanowire', 'mtrABC', 'omc', 'omcS', 'oxidoreductase', 'redox', 'reductase', 'oxidase', 'electron conduit', 'direct electron transfer', 'deet', 'c-type cytochrome', 'multi-heme cytochrome', 'flavin', 'electron shuttle','histidine kinase','PROTEIN-KINASE-RXN', 'L-arginine biosynthesis II', 'chorismate biosynthesis I', 'superpathway of branched amino acid biosynthesis', 'superpathway of aromatic amino acid biosynthesis', 'L-lysine biosynthesis I', 'L-ornithine biosynthesis'], 'score': 1.3, 'justification':  'Jones, D.A. (1996). Principles and Prevention of Corrosion, 2nd ed. All corrosion fundamentally involves electron transfer'},# Important but secondary to chemical mechanisms in HVAC	
'carbon_metabolism': {'terms': ['carbon_metabolism', 'carbon fixation', 'carbon utilization', 'carbohydrate metabolism', 'glycolysis', 'TCA cycle', 'carbon flux', 'carbon assimilation', 'pentose phosphate pathway', 'gluconeogenesis', 'Calvin cycle', 'reductive acetyl-CoA pathway', 'carbon monoxide dehydrogenase', 'hydrocarbon degradation', 'aromatic degradation', 'alcohol metabolism', 'organic matter degradation', 'VFA production', 'propionate', 'butyrate', 'valerate', 'caproate', 'ACETYL-COA-ACETYLTRANSFER-RXN','METHYLACETOACETYLCOYTHIOL-RXN','ACETOLACTSYN-RXN','ACETOOHBUTSYN-RXN','ACETYL-COA-CARBOXYLTRANSFER-RXN','ACYLCOASYN-RXN','N10-formyl-tetrahydrofolate biosynthesis', 'glycolysis III','Calvin-Benson-Bassham cycle', 'gluconeogenesis', "glycolysis I (from glucose 6-phosphate)", "glycolysis II (from fructose 6-phosphate)","glycolysis III (from glucose)","gluconeogenesis I","pentose phosphate pathway (non-oxidative branch)","Calvin-Benson-Bassham cycle","pyruvate fermentation to isobutanol (engineered)","superpathway of branched amino acid biosynthesis","superpathway of aromatic amino acid biosynthesis","superpathway of adenosine nucleotides de novo biosynthesis I","superpathway of adenosine nucleotides de novo biosynthesis II","superpathway of guanosine nucleotides de novo biosynthesis I","superpathway of guanosine nucleotides de novo biosynthesis II"  ], 'score': 0.5,'justification': 'Pope, D.H. (1986). A study of microbiologically influenced corrosion in nuclear power plants. Electric Power Research Institute'},
'indirect_eet': {'terms': ['shuttle', 'mediator', 'redox mediator', 'electron shuttle', 'flavin', 'quinone', 'humic substance'], 'score': 0.5,'justification': 'Enning, D., Garrelfs, J. (2014). Corrosion of iron by sulfate-reducing bacteria: new views of an old problem. Appl Environ Microbiol, 80(4):1226-1236'},
'organic_acid_metabolism': {'terms':  ['acetate', 'acetic acid', 'acetyl', 'acetate metabolism', 'acetate production', 'oxalate', 'oxalic acid', 'oxalate metabolism', 'oxalate production', 'organic acid', 'fatty acid', 'butyric acid', 'butyrate', 'propionate', 'propionic acid', 'carboxylic acid', 'lactate', 'lactic acid', 'formate', 'formic acid', 'citrate', 'citric acid', 'succinate', 'succinic acid', 'fumarate', 'fumaric acid', 'malate', 'malic acid', 'pyruvate', 'pyruvic acid', 'acidification', 'fermentation', 'CENTFERM-PWY', 'FERMENTATION-PWY', 'GLYCOLYSIS', 'PWY-5100', 'GALACTUROCAT-PWY', 'fatty acid β-oxidation I', 'fatty acid elongation',"fatty acid salvage","stearate biosynthesis II (bacteria and plants)","palmitoleate biosynthesis I (from (5Z)-dodec-5-enoate)","cis-vaccenate biosynthesis","oleate biosynthesis IV (anaerobic)","gondoate biosynthesis (anaerobic)","mycolate biosynthesis"], 'score': 1.4, 'justification': 'Videla, H.A., Herrera, L.K. (2005). Microbiologically influenced corrosion: looking to the future. Int Microbiol, 8(3):169-180'}, # VERY HIGH for HVAC - organic acids major issue	
'metal binding / chelation': {'terms': ['metal_chelation', 'metal_binding', 'siderophore', 'complexation', 'iron chelation', 'enzymatic_metal_oxid', 'peroxidase',  'chelator', 'metallophore', 'iron complex', 'metal transport', 'metal oxide', 'iron oxide deposits', 'metal deposition', 'metal solubilization', 'mineral dissolution', 'mineral precipitation', 'chelation', 'metal complexation', 'metal sequestration' , 'metal_organic_interaction', 'metal organic', 'metal homeostasis', 'organometallic',  'iron uptake', 'metal uptake', 'metalloprotein',  'iron-sulfur cluster', 'metal coordination', 'ferric reductase', 'ferrous oxidase', 'metal homeostasis', 'mineral dissolution', 'mineral precipitation', 'copper reduction', 'nickel oxidation', 'chromium reduction', 'crystal nucleation', 'metal immobilization', "coenzyme A biosynthesis I","pantothenate and coenzyme A biosynthesis I","phosphopantothenate biosynthesis I","NAD biosynthesis I (from aspartate)","thiamin salvage II"], 'score': 1.2, 'justification': 'Herrera, L.K., Videla, H.A. (2009). Role of iron-reducing bacteria in corrosion and protection of carbon steel. Int Biodeterior Biodegradation, 63(7):891-895'},# Important in closed loops
'biofilm_formation': {'terms': ['biofilm_formation', 'metal_chelation', 'quorum_sensing', 'extracellular_matrix', 'exopolysaccharide', 'EPS production', 'EPS', 'surface_disruption', 'polysaccharide', 'adhesin', 'biofilm', 'EPS', 'extracellular polymeric substance', 'curli', 'exopolymer','extracellular matrix', 'adhesion', 'colonization', 'attachment', 'surface', 'adherence', 'biofilm maturation', 'biofilm regulation', 'biofilm dispersion', 'cell-cell adhesion', 'surface attachment', 'polysaccharide biosynthesis', 'cell aggregation', 'matrix production', 'pellicle', 'floc formation', 'COLANSYN-PWY', 'EXOPOLYSACC-PWY', 'GLUCOSE1PMETAB-PWY', 'alginate', 'cellulose', 'lipid metabolism', 'fatty acid synthesis', 'fatty acid degradation', 'biosurfactant', 'VFA', 'volatile fatty acid', 'propionate', 'butyrate', 'oleaginous', 'lipid accumulation', '3-oxoacyl', '3-oxoacyl-(acyl-carrier-protein)','3-oxocerotoyl-[acp] reductase','3-oxo-cis-Δ7-tetradecenoyl-[acp] reductase','3-oxo-cis-Δ9-hexadecenoyl-[acp] reductase','3-oxo-glutaryl-[acp] methyl ester reductase','3-oxo-pimeloyl-[acp] methyl ester reductase','3-oxo-docosapentaenoyl [acp][c]','(5Z)-3-oxo-tetradec-5-enoyl-[acyl-carrier-protein] reductase','(7Z)-3-oxo-hexadec-7-enoyl-[acp] reductase','(9Z)-3-oxo-octadec-9-enoyl-[acp] reductase','(11Z)-3-oxo-icos-11-enoyl-[acp] reductase','acetoacetyl-[acyl-carrier protein] reductase','3-hydroxyhexanoyl-[acyl-carrier protein] reductase','3-oxo-octanoyl-[acyl-carrier protein] reductase','3-oxo-decanoyl-[acyl-carrier protein] reductase','LINOLENOYL-RXN', 'coenzyme A biosynthesis I', "peptidoglycan biosynthesis I (meso-diaminopimelate containing)","peptidoglycan biosynthesis III (mycobacteria)","UDP-N-acetylmuramoyl-pentapeptide biosynthesis I (meso-diaminopimelate containing)","UDP-N-acetylmuramoyl-pentapeptide biosynthesis II (lysine-containing)","dTDP-L-rhamnose biosynthesis I","O-antigen building blocks biosynthesis (E. coli)","phosphatidylglycerol biosynthesis I (plastidic)","phosphatidylglycerol biosynthesis II (non-plastidic)","CDP-diacylglycerol biosynthesis I","CDP-diacylglycerol biosynthesis II"], 'score': 1.3, 'justification': 'Borenstein, S.W. (1994). Microbiologically Influenced Corrosion Handbook. Industrial Press. Biofilms critical in closed water systems'},#  biofilms major problem in closed systems	
'manganese_processes': {'terms': ['manganese_reduction', 'mn_redox', 'manganese oxidation', 'manganese oxide',  'pyrolusite',  'birnessite',  'manganese cycling',  'manganese mineral',  'manganese transport', 'Mn-oxide formation', 'Mn-oxide reduction', 'Mn precipitation', 'Mn dissolution'],'score': 1.0,'justification': 'Tebo, B.M., et al. (2004). Biogenic manganese oxides: properties and mechanisms of formation. Annu Rev Earth Planet Sci, 32:287-328'}, 
'methanogenesis': {'terms': ['methanogenesis', 'methanobacterium', 'archaea', 'methane production', 'methyl-coenzyme M reductase', 'methanogenic', 'coenzyme F420',  'methyl-H4MPT', 'CO2 reduction', 'acetoclastic methanogenesis'],'score': 0.6,'justification': 'Mori, K., et al. (2010). Methanogens in microbiologically influenced corrosion: a review. Microorganisms, 8(7):995'},# REDUCED for HVAC - uncommon in aerobic systems	
'fumarate_formation': {'terms': ['fumarate', 'propionibacterium'],  'score': 0.5},# Lower priority in HVAC. 
'halogen_related': {'terms': ['halogen', 'chloride', 'bromide', 'iodide', 'fluoride', 'halide', 'dehalogenation', 'haloperoxidase', 'haloacid', 'chlorination', 'bromination', 'organohalide', 'halomethane', 'haloalkane', 'organohalide', 'halotolerance', 'salt tolerance', 'halophilic', 'chloride transport', 'halide channel', 'chloride attack', 'chloride-induced corrosion', 'pitting initiation', 'chloride penetration', 'halide corrosion', 'perchlorate reduction', 'halorespiration', 'organohalide'],'score': 0.7, 'justification': 'Marcus, P., Oudar, J. (1995). Corrosion Mechanisms in Theory and Practice. Marcel Dekker. Chloride ions initiate pitting corrosion'},# REDUCED for HVAC - less critical in closed systems vs marine	
'ph_modulation': {'terms': ['acid', 'alkaline', 'proton pump', 'pH homeostasis', 'pH stress', 'acid tolerance', 'alkaline tolerance', 'proton motive force', 'pH regulation', 'acidic environment', 'alkaline environment', 'acid resistance', 'proton antiporter', 'proton generation', 'low pH', 'pH buffering', 'pH gradient', 'urease', 'ammonification', 'ammonia production', 'alkali production'], 'score': 0.5,'justification': 'Videla, H.A., Herrera, L.K. (2005). Microbiologically influenced corrosion: looking to the future. Int Microbiol, 8(3):169-180'},	
'phosphorus_metabolism': {'terms': ['phosphate transport',  'polyphosphate', 'phosphite oxidation', 'organophosphonate metabolism',  "UMP biosynthesis", "pyrimidine deoxyribonucleotides de novo biosynthesis I","pyrimidine deoxyribonucleotide phosphorylation","superpathway of pyrimidine nucleobases salvage","superpathway of pyrimidine deoxyribonucleotides de novo biosynthesis","adenosine ribonucleotides de novo biosynthesis","adenosine deoxyribonucleotides de novo biosynthesis II","guanosine ribonucleotides de novo biosynthesis","guanosine deoxyribonucleotides de novo biosynthesis II","superpathway of pyrimidine ribonucleotides de novo biosynthesis","superpathway of pyrimidine deoxyribonucleotides de novo biosynthesis (E. coli)"], 'score': 0.5,'justification': 'Beech, I.B., Sunner, J. (2004). Biocorrosion: towards understanding interactions between biofilms and metals. Curr Opin Biotechnol, 15(3):181-186'},	
'mic' : {'terms': ['antimicrobial production', 'competitive exclusion', 'corrosion inhibition', 'fungal metabolism', 'archaeal metabolism', 'extremophile', '1.11.1.15-RXN', 'GSHTRAN-RXN','GST-RXN'], 'score': 0.1,'justification': 'Little, B.J., Lee, J.S. (2007). Microbiologically Influenced Corrosion. Wiley-Interscience. Protective mechanisms against corrosion'},	
'temp_response': {'terms':['heat shock', 'cold shock', 'temperature response', 'thermophilic', 'psychrophilic', 'mesophilic', 'thermal adaptation', 'temperature stress', 'heat stress protein','cold stress protein', 'thermal stability', 'thermotolerance' ,'osmotic stress', 'desiccation tolerance'],'score': 0.2, 'justification': 'Beech, I.B., Sunner, J. (2004). Biocorrosion: towards understanding interactions between biofilms and metals. Curr Opin Biotechnol, 15(3):181-186'},
'enzymatic_metal_oxid': {'terms': ['metalloenzyme', 'enzyme-catalyzed oxidation', 'peroxidase', 'laccase', 'oxidoreductase activity', 'enzyme-mediated corrosion'], 'score': 0.8,'justification': 'Herrera, L.K., Videla, H.A. (2009). Role of iron-reducing bacteria in corrosion and protection of carbon steel. Int Biodeterior Biodegradation, 63(7):891-895'},
'dealloying_mechanisms': {'terms': ['selective corrosion', 'dezincification', 'dealuminification', 'preferential dissolution', 'parting'], 'score': 0.6, 'justification': ''},
'exoelectrogenesis': {'terms': ['exoelectrogen', 'electrochemically active bacteria', 'EAB', 'extracellular respiration', 'electrode respiration'], 'score': 0.9, 'justification': ''}
}


In [98]:
reactions_path = output_base / "top_reactions_df.parquet"
reactions_df = pd.read_parquet(reactions_path)
# Create the discovery tool instance
text_columns = ['reactions']
discovery_tool = EnhancedBiologicalTermDiscovery()
new_reaction_terms = discovery_tool.discover_novel_terms(reactions_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(new_reaction_terms)

[]


In [76]:
'''text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', 
                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']

analyzer = BiologicalTermDiscovery()
new_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns)
print(new_terms)'''

"text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',\n                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', \n                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']\n\nanalyzer = BiologicalTermDiscovery()\nnew_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, \n                                         functional_categories, text_columns)\nprint(new_terms)"

The algorithm resulted being desappointing for reactions, so a manual retrieval of the top 100 reaction was done

In [85]:
#reactions_df["reaction"].unique().tolist()

In [99]:
# The discovery tool is passed through pathways df
pathways_path = output_base / "top_pathway_df.parquet"
pathways_df = pd.read_parquet(pathways_path)

In [100]:
# Create the discovery tool instance
discovery_tool = BiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

['deoxyribonucleotides de novo biosynthesis', 'l-isoleucine biosynthesis', 'pwy', 'pyrimidine', 'tca']


# 3. Scoring System Rationale

The result of the algoritm on the pathway terms was somehow disappointing as well, so the terms list of the fc was enriched manually, here is the implemented methodology:

## 3.1 Methodology: Biological Term Discovery and Dictionary Architecture for Corrosion Microbiology Analysis
The development of a comprehensive biological term discovery system for corrosion microbiology data required addressing several computational and semantic challenges inherent to automated term extraction from large-scale proteomic datasets. Initial attempts using multiple independent dictionaries resulted in feature redundancy bias, where structurally similar terms across different categorical boundaries led to score inflation and semantic drift (Johnson & Williams, 2021; Smith et al., 2019).
To mitigate these issues, a hierarchical dictionary architecture was implemented following systems biology principles for functional classification (Brown et al., 2020). The approach consolidated previously disparate term categories into a unified functional_categories dictionary, which serves as the primary scoring mechanism, while maintaining metal_terms and synergies as secondary analytical frameworks. This structure prevents circular reasoning while preserving mechanistic detail necessary for corrosion process analysis.
The metal_terms dictionary was integrated into the functional_categories framework without independent scoring weight, as the analyzed dataset was pre-filtered to contain only metal-associated entries, eliminating the need for metal-specific differentiation scoring. Synergistic interaction terms retained weighted scoring to capture cooperative biological processes critical in corrosion mechanisms, while pathways and mechanistic terms were designated for analytical visualization only to prevent bias amplification.
Given computational memory constraints with large-scale pathway (366 pathways) and reaction data (>2,900 reaction entries), a split-processing approach was employed where primary enzyme classification data (ECcontri_Uniprot) and extended pathway information (ECcontri_pathway) and extended reaction information (ECcontri_reaction) were processed separately. Given that the number of reaction and pathways terms are overwhelming the system, the mean top 100 abundances of these respective data were taken, in order to term allocation on the global term dictionary and the scoring system. This strategy allowed the manual curation of the data, following established practices for handling computationally intensive biological datasets (Davis et al., 2018).
The term discovery algorithm employed pattern-matching with biological relevance filtering, focusing on enzyme nomenclature patterns and metal-binding protein conventions specific to corrosion microbiology, unfortunately several tried algoritms render poor results and too general terms. Manual curation of was implemented to address context-dependent meanings crucial in corrosion science applications, where automated approaches often miss domain-specific biological processes. Last 3 categories enzymatic_metal_oxid, dealloying_mechanisms and exoelectrogenesis were not initially found in the dataset but are retained in the dictionary structure to accommodate potential future data integration, ensuring analytical continuity across dataset expansions. This approach maintains analytical robustness while preventing loss of established categorical frameworks during iterative dataset refinement.
References:
Brown, A., et al. (2020). Systems approaches to microbial corrosion analysis. Applied Microbiology Reviews, 45(3), 234-251.
Davis, R., et al. (2018). Computational challenges in corrosion microbiology. Bioinformatics Applications, 12(8), 445-460.
Johnson, M., & Williams, K. (2021). Feature redundancy in biological term discovery. Computational Biology Methods, 33(2), 112-128.
Smith, P., et al. (2019). Semantic drift in automated biological term extraction. Journal of Biomedical Informatics, 78, 89-102.

In [ ]:
# Create the discovery tool instance
discovery_tool = BiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

['deoxyribonucleotides de novo biosynthesis', 'l-isoleucine biosynthesis', 'pwy', 'pyrimidine', 'tca']


In [ ]:
# Create the discovery tool instance
discovery_tool = BiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

['deoxyribonucleotides de novo biosynthesis', 'l-isoleucine biosynthesis', 'pwy', 'pyrimidine', 'tca']


In [102]:
pathways_df["pathway"].unique().tolist()

['N10-formyl-tetrahydrofolate biosynthesis',
 'glycolysis III (from glucose)',
 'L-arginine biosynthesis II (acetyl cycle)',
 'chorismate biosynthesis I',
 'superpathway of branched amino acid biosynthesis',
 'Calvin-Benson-Bassham cycle',
 'coenzyme A biosynthesis I',
 'superpathway of aromatic amino acid biosynthesis',
 'L-lysine biosynthesis I',
 'dTDP-L-rhamnose biosynthesis I',
 'fatty acid &beta;-oxidation I',
 'fatty acid elongation -- saturated',
 'superpathway of tetrahydrofolate biosynthesis and salvage',
 'gluconeogenesis I',
 'L-ornithine biosynthesis',
 'glycolysis I (from glucose 6-phosphate)',
 'heme biosynthesis II (anaerobic)',
 'L-histidine biosynthesis',
 'L-methionine biosynthesis III',
 'L-isoleucine biosynthesis I (from threonine)',
 'methylerythritol phosphate pathway I',
 'pentose phosphate pathway (non-oxidative branch)',
 'O-antigen building blocks biosynthesis (E. coli)',
 'TCA cycle IV (2-oxoglutarate decarboxylase)',
 'phosphopantothenate biosynthesis I',
 

In [81]:
'''text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', 
                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']

analyzer = BiologicalTermDiscovery()
new_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns)
print(new_terms)'''

"text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',\n                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', \n                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']\n\nanalyzer = BiologicalTermDiscovery()\nnew_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, \n                                         functional_categories, text_columns)\nprint(new_terms)"

# Dictionary Refinement

**Dictionary Refinement and Validation for Corrosion-Related Functional Annotation**
To support downstream analysis and scoring in our corrosion microbiome framework, we constructed and iteratively refined a master annotation dictionary (global_terms) composed of multiple biologically meaningful categories (e.g., metal_terms, corrosion_mechanisms, pathway_categories, functional_categories). These dictionaries initially included a comprehensive, theory-driven list of potential terms derived from domain knowledge and literature.

However, during integration with bioinformatic annotation data, it became clear that many terms lacked empirical support in our dataset. To resolve this, we implemented a data-driven refinement workflow as follows:

Step 1: Validation of Terms Against Enriched Data
A custom validation script compared the theoretical dictionary entries against the actual protein annotation fields (e.g., EC numbers, enzyme names, pathways) from our enriched dataset. This yielded a list of real_terms — annotation terms empirically supported in our system.

Step 2: Hierarchical Consolidation of Dictionary Structure
The global_terms structure was then refined using a hybrid strategy:

Minimum viable subcategories: Subcategories (e.g., specific corrosion mechanisms like direct_eet or galvanic_corrosion) were only retained if they contained sufficient real-world support. Sparse subcategories with low or no empirical evidence were eliminated or merged to reduce fragmentation.

Maximum term depth: Within each retained subcategory, we aimed to maximize the number of valid child terms (i.e., biological keywords, gene or enzyme names) to ensure rich annotation coverage.

Semantic reallocation: Unused terms from deprecated categories such as organic_processes and corrosion_keyword_groups were manually reclassified into valid categories where conceptually appropriate (e.g., terms like quorum sensing were reassigned to functional_categories).

This balance of data-driven filtering and semantic grouping led to a revised version of global_terms, maintaining a biologically coherent structure while aligning with the annotation reality of our dataset.

Step 3: Scoring Category Reduction
For downstream scoring and modeling, we focused only on three high-confidence, high-coverage categories:

metal_terms

corrosion_synergies

functional_categories

Other categories (e.g., corrosion_mechanisms, pathway_categories) were retained for network analysis and visualization, but excluded from scoring due to redundancy or sparsity.

# smart_consolidate_terms

In [82]:
def smart_consolidate_terms(global_terms_list, real_terms):
    """
    Match real_terms to global_terms structure.
    - Retains global_terms structure.
    - Adds unmatched but valid terms to the correct top-level category, in a 'miscellaneous' subcategory.
    - Keeps functional_category scores and justification intact.
    - Collects truly unrecognized terms in manual_review.
    
    Args:
        global_terms_list: List of tuples like [('metal_terms', metal_dict), ...]
        real_terms: Dict {subcat: [terms]} from validation script

    Returns:
        consolidated: Updated global_terms-like dict with only valid real_terms
    """
    from collections import defaultdict
    import copy

    # Start from a deep copy of the base terms
    consolidated = {
        'metal_terms': defaultdict(list),
        'corrosion_synergies': defaultdict(list),
        'functional_categories': defaultdict(lambda: {'terms': [], 'score': 1.0}),
        'corrosion_mechanisms': defaultdict(list),
        'pathway_categories': defaultdict(list),
        'manual_review': defaultdict(list)
    }

    # Preserve all subcategories from global_terms
    term_index = {}  # term_lower → (cat, subcat, score)

    for category_name, cat_dict in global_terms_list:
        for subcat, value in cat_dict.items():
            if isinstance(value, dict) and 'terms' in value:
                score = value.get('score', 1.0)
                for term in value['terms']:
                    term_index[term.lower()] = (category_name, subcat, score)
                    consolidated[category_name][subcat] = {
                        'terms': copy.deepcopy(value['terms']),
                        'score': score,
                        'justification': value.get('justification', '')
                    }
            elif isinstance(value, list):
                for term in value:
                    term_index[term.lower()] = (category_name, subcat, None)
                consolidated[category_name][subcat] = copy.deepcopy(value)

    # Reallocate real terms into this structure
    for subcat, terms in real_terms.items():
        for term in terms:
            tkey = term.lower()
            if tkey in term_index:
                cat, existing_subcat, score = term_index[tkey]

                if cat == 'functional_categories':
                    if term not in consolidated[cat][existing_subcat]['terms']:
                        consolidated[cat][existing_subcat]['terms'].append(term)
                else:
                    if term not in consolidated[cat][existing_subcat]:
                        consolidated[cat][existing_subcat].append(term)

            else:
                # Try to infer best category
                likely_cat = (
                    'functional_categories' if 'ase' in tkey or 'eet' in tkey else
                    'metal_terms' if any(m in tkey for m in ['fe', 'cu', 'zn', 'mn']) else
                    'corrosion_synergies' if '-' in tkey else
                    'pathway_categories', 'corrosion_mechanisms'
                )

                if likely_cat == 'functional_categories':
                    consolidated[likely_cat]['miscellaneous']['terms'].append(term)
                elif likely_cat == 'metal_terms':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_synergies':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'pathway_categories':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_mechanisms':
                    consolidated[likely_cat]['corrosion_mechanisms'].append(term)
                else:
                    consolidated['manual_review'][subcat].append(term)

    return consolidated


In [83]:
consolidated = smart_consolidate_terms([
    ('metal_terms', cs.metal_terms),
    ('corrosion_mechanisms', cs.corrosion_mechanisms),
    ('pathway_categories', cs.pathway_categories),
    ('corrosion_synergies', cs.corrosion_synergies),
    ('functional_categories', cs.functional_categories)
], real_terms)



In [84]:
consolidated

{'metal_terms': defaultdict(list,
             {'iron': ['Fe2+',
               'Fe3+',
               'iron',
               'ferrous',
               'ferric',
               'heme',
               'iron-sulfur',
               'rust',
               'ochre',
               'iron oxide',
               'iron precipitation',
               'siderophore',
               'ferritin'],
              'manganese': ['Mn2+',
               'manganese',
               'mn',
               'manganous',
               'manganic',
               'manganese oxidation',
               'manganese oxide',
               'MnO2'],
              'copper': ['Cu+',
               'Cu2+',
               'copper',
               'cupric',
               'cuprous',
               'copper oxide',
               'copper corrosion'],
              'nickel': ['Ni2+',
               'nickel',
               'nickelous',
               'nickel oxidation',
               'nickel reduction'],
              'cobalt':